# **Cyber Bullying Detection**

By Using GapHate Corpus we imported data of catigorized tweets to train the data:

-vo : violent language or context

-hd : hate speech

-cv : esplicit calls to violence

In [84]:
pip install pyspark

In [85]:
import pandas as pd

train = pd.read_csv('ghc_train.tsv', sep='\t')
test = pd.read_csv('ghc_test.tsv', sep='\t')

spaCy for to clean text

In [86]:
import spacy
from pyspark.sql.functions import udf
from pyspark.sql.types import ArrayType, StringType

nlp = spacy.load("en_core_web_sm")

def lemmatize_spacy(words):
    doc = nlp(" ".join(words))
    return [token.lemma_ for token in doc]
def preprocess(text):
    doc = nlp(text)
    return " ".join([
        token.lemma_
        for token in doc
        if not token.is_stop and token.is_alpha
    ])
lemma_udf = udf(lemmatize_spacy, ArrayType(StringType()))

train['lemmatized'] = train['text'].apply(preprocess)

In [87]:
train.to_csv("train_clean.csv", index=False)
test.to_csv("test_clean.csv", index=False)

Distributed Load

In [88]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Cyberbullying").getOrCreate()

train_spark = spark.read.csv("train_clean.csv", header=True, inferSchema=True)
test_spark = spark.read.csv("test_clean.csv", header=True, inferSchema=True)

Cleaning Columns

In [89]:
from pyspark.sql.functions import col

train_spark = train_spark.filter(col("lemmatized").isNotNull())
train_spark = train_spark.filter(col("lemmatized") != "")

Pipeline

In [90]:
from pyspark.sql.functions import expr

train_spark = train_spark.withColumn("hd", expr("try_cast(hd as int)"))
train_spark = train_spark.withColumn("cv", expr("try_cast(cv as int)"))
train_spark = train_spark.withColumn("vo", expr("try_cast(vo as int)"))

train_spark = train_spark.dropna(subset=["hd", "cv", "vo"])

In [91]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import Tokenizer, NGram, HashingTF, IDF

tokenizer = Tokenizer(inputCol="lemmatized", outputCol="words")
ngram = NGram(n=2, inputCol="words", outputCol="bigrams")
hashingTF = HashingTF(inputCol="bigrams", outputCol="rawFeatures")
idf = IDF(inputCol="rawFeatures", outputCol="features")

pipeline = Pipeline(stages=[tokenizer, ngram, hashingTF, idf])
model = pipeline.fit(train_spark)


train_spark = model.transform(train_spark)

In [92]:
train_spark.printSchema()

root
 |-- text: string (nullable = true)
 |-- hd: integer (nullable = true)
 |-- cv: integer (nullable = true)
 |-- vo: integer (nullable = true)
 |-- lemmatized: string (nullable = true)
 |-- words: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- bigrams: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- rawFeatures: vector (nullable = true)
 |-- features: vector (nullable = true)



labels>int

# **Training the Model**

In [93]:
train_spark.select("text", "hd", "cv", "vo").show(5, truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---+---+---+
|text                                                                                                                                                                                                                                                                                                                                                                       |hd |cv |vo |
+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [94]:
from pyspark.ml.classification import LogisticRegression

lr_hd = LogisticRegression(featuresCol="features", labelCol="hd")
model_hd = lr_hd.fit(train_spark)

In [95]:
lr_vo = LogisticRegression(featuresCol="features", labelCol="vo")
model_vo = lr_vo.fit(train_spark)

In [96]:
lr_cv = LogisticRegression(featuresCol="features", labelCol="cv")
model_cv = lr_cv.fit(train_spark)

In [97]:
pred_hd = model_hd.transform(train_spark)
pred_hd.select("hd", "prediction").show()

+---+----------+
| hd|prediction|
+---+----------+
|  0|       0.0|
|  0|       0.0|
|  0|       0.0|
|  0|       0.0|
|  0|       0.0|
|  0|       0.0|
|  0|       0.0|
|  0|       0.0|
|  0|       0.0|
|  0|       0.0|
|  0|       0.0|
|  0|       0.0|
|  0|       0.0|
|  0|       0.0|
|  0|       0.0|
|  0|       0.0|
|  0|       0.0|
|  0|       0.0|
|  0|       0.0|
|  0|       0.0|
+---+----------+
only showing top 20 rows


Evaluation


In [98]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

evaluator = BinaryClassificationEvaluator(labelCol="hd")

print("HD accuracy:", evaluator.evaluate(pred_hd))

HD accuracy: 0.9995917507540545


In [100]:
pred_vo = model_vo.transform(train_spark)
pred_vo.select("vo", "prediction").show()

evaluator = BinaryClassificationEvaluator(labelCol="vo")

print("VO accuracy:", evaluator.evaluate(pred_vo))

+---+----------+
| vo|prediction|
+---+----------+
|  0|       0.0|
|  0|       0.0|
|  0|       0.0|
|  0|       0.0|
|  0|       0.0|
|  0|       0.0|
|  0|       0.0|
|  0|       0.0|
|  0|       0.0|
|  0|       0.0|
|  0|       0.0|
|  0|       0.0|
|  0|       0.0|
|  0|       0.0|
|  0|       0.0|
|  0|       0.0|
|  0|       0.0|
|  0|       0.0|
|  0|       0.0|
|  0|       0.0|
+---+----------+
only showing top 20 rows
VO accuracy: 0.9994943369776037


In [101]:
pred_cv = model_cv.transform(train_spark)
pred_cv.select("cv", "prediction").show()

evaluator = BinaryClassificationEvaluator(labelCol="cv")

print("cv accuracy:", evaluator.evaluate(pred_cv))

+---+----------+
| cv|prediction|
+---+----------+
|  0|       0.0|
|  0|       0.0|
|  0|       0.0|
|  0|       0.0|
|  0|       0.0|
|  0|       0.0|
|  0|       0.0|
|  0|       0.0|
|  0|       0.0|
|  0|       0.0|
|  0|       0.0|
|  0|       0.0|
|  0|       0.0|
|  0|       0.0|
|  0|       0.0|
|  0|       0.0|
|  0|       0.0|
|  0|       0.0|
|  0|       0.0|
|  0|       0.0|
+---+----------+
only showing top 20 rows
cv accuracy: 0.9999940334128878


Testing on another data set